# MiniReasoner in Google Colab

Prepare SimpleStories, train the tokenizer and the decoder-only Transformer, then generate text from the saved checkpoint. All model and training code is included below.

1. For training, choose **Runtime > Change runtime type**, then select a GPU. For CPU inference, select a CPU runtime and set **train_model = False**.
2. For authenticated downloads, add a Hugging Face read token to **Secrets** as **HF_TOKEN** and enable notebook access. The public dataset can also be downloaded anonymously.
3. Run the cells from top to bottom. The default is the 32M-parameter base model and up to 60,000 training stories plus 1,024 validation stories.

The first run downloads about 1.7 GB of published training shards before selecting the local stories. Local validation is held out from those shards, and the published test split is reserved for final evaluation.

[Google Colab FAQ](https://research.google.com/colaboratory/faq.html) | [SimpleStories dataset](https://huggingface.co/datasets/SimpleStories/SimpleStories)

## Install notebook dependencies

Use uv to install the data libraries into the active Colab Python environment. Training uses the CUDA-enabled PyTorch already in the Colab runtime.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

subprocess.run(
    [
        sys.executable, "-m", "uv", "pip", "install",
        "--python", sys.executable, "datasets>=3,<5", "tokenizers>=0.20,<1",
    ],
    check=True,
)

## Run settings

Training requires a CUDA GPU. Use **base** for pretraining or **small** for a shorter GPU run. The story limits control data size independently. Automatic precision uses BF16 on supported GPUs and FP32 on GPUs such as the T4.

By default, files live under **/content/MiniReasoner** and disappear when Colab deletes the runtime. Set **use_drive = True** before your first run to keep the packed data and checkpoint in **MyDrive/MiniReasoner**. The raw dataset cache stays on the Colab runtime disk.

In [ ]:
import json
import os
from dataclasses import asdict
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter

import torch

# Settings
preset = "base"
train_model = True
inference_device = "cpu"
precision_request = "auto"
use_drive = False
stop_after = None
resume = False

VOCAB_SIZE = 1_024
story_limits = {"train": 60_000, "val": 1_024}
seed = 42
dataset_name = "SimpleStories/SimpleStories"
SIMPLESTORIES_TRAIN_FILES = "data/train-*.parquet"
source_splits = {"train": "train", "val": "train"}

runtime_dir = Path("/content/MiniReasoner")
workspace_dir = runtime_dir
if use_drive:
    from google.colab import drive

    drive.mount("/content/drive")
    workspace_dir = Path("/content/drive/MyDrive/MiniReasoner")

cache_dir = runtime_dir / "data" / "raw"
tokenized_path = workspace_dir / "data" / "tokenized"
model_dir_name = "simplestories" if preset == "base" else "simplestories-small"
run_dir = workspace_dir / "models" / model_dir_name
checkpoint_path = run_dir / "base.pt"

PRESETS = {
    "base": {
        "model": {
            "context_length": 256,
            "n_layer": 10,
            "n_head": 8,
            "n_embd": 512,
        },
        "training": {
            "batch_size": 64,
            "steps": 1500,
            "warmup_steps": 50,
            "learning_rate": 0.0005,
            "min_learning_rate": 0.00005,
            "weight_decay": 0.1,
            "eval_every": 100,
            "eval_windows": 64,
            "seed": 42,
        },
    },
    "small": {
        "model": {
            "context_length": 64,
            "n_layer": 2,
            "n_head": 4,
            "n_embd": 64,
        },
        "training": {
            "batch_size": 32,
            "steps": 60,
            "warmup_steps": 5,
            "learning_rate": 0.001,
            "min_learning_rate": 0.0001,
            "weight_decay": 0.1,
            "eval_every": 20,
            "eval_windows": 32,
            "seed": 42,
        },
    },
}

In [ ]:
def require_training_gpu() -> torch.device:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "Training requires a CUDA GPU. Select a GPU runtime or set train_model=False for inference."
        )
    return torch.device("cuda")


def resolve_precision(request: str, device: torch.device) -> str:
    if request == "fp32":
        return "fp32"
    if request not in ("auto", "bf16"):
        raise ValueError("Training precision must be auto, fp32, or bf16")
    native_bf16 = torch.cuda.get_device_capability(
        device)[0] >= 8 and torch.cuda.is_bf16_supported()
    if request == "auto":
        return "bf16" if native_bf16 else "fp32"
    if not native_bf16:
        raise ValueError("This CUDA device does not have native BF16 support")
    return "bf16"


if train_model:
    device = require_training_gpu()
    precision = resolve_precision(precision_request, device)
    print(f"Runtime: {torch.cuda.get_device_name(device)} | {precision}")

## Tokenizer and packed data helpers

The byte-level BPE tokenizer learns from the local training stories. Digits remain separate, and every packed story ends with an EOS token.

In [ ]:
from collections.abc import Iterable, Iterator
from hashlib import file_digest
from itertools import chain, islice

import numpy as np
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers


EOS = "<|eos|>"
TOKEN_DTYPE = np.dtype("<u2")


def train_tokenizer(documents: Iterable[str], vocab_size: int) -> Tokenizer:
    if not 257 <= vocab_size <= 65_536:
        raise ValueError("Vocabulary size must be between 257 and 65,536")
    tokenizer = Tokenizer(models.BPE())
    byte_level = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.pre_tokenizer = pre_tokenizers.Sequence(
        [pre_tokenizers.Digits(individual_digits=True), byte_level]
    )
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        initial_alphabet=byte_level.alphabet(),
        special_tokens=[EOS],
        show_progress=False,
    )
    tokenizer.train_from_iterator(documents, trainer=trainer)
    return tokenizer


def sha256(path: Path) -> str:
    with path.open("rb") as stream:
        return file_digest(stream, "sha256").hexdigest()


def pack_documents(path: Path, tokenizer: Tokenizer, documents: Iterator[str]) -> dict:
    eos_id = tokenizer.token_to_id(EOS)
    tokens = examples = 0
    with path.open("wb") as stream:
        while batch := list(islice(documents, 2048)):
            encoded = tokenizer.encode_batch(batch, add_special_tokens=False)
            token_ids = chain.from_iterable(chain(item.ids, (eos_id,)) for item in encoded)
            packed = np.fromiter(token_ids, dtype=TOKEN_DTYPE)
            stream.write(packed.tobytes())
            tokens += len(packed)
            examples += len(batch)
    return {"documents": examples, "tokens": tokens, "sha256": sha256(path)}

## Download and inspect SimpleStories

Download the published training shards into the raw cache, shuffle with a fixed seed, and reserve the validation stories first. Exact copies of validation stories are excluded from training. If the packed corpus is already present with these settings, reuse it.

In [ ]:
if train_model:
    output_files = ("train.bin", "val.bin", "tokenizer.json", "manifest.json")
    corpus_settings = {
        "dataset": dataset_name,
        "source_splits": source_splits,
        "seed": seed,
        "story_limits": story_limits.copy(),
        "target_vocab_size": VOCAB_SIZE,
    }

    present_files = [name for name in output_files if (tokenized_path / name).is_file()]
    if present_files and len(present_files) != len(output_files):
        raise FileExistsError(
            f"Packed data is incomplete in {tokenized_path}. "
            "Move the incomplete folder aside or choose a new workspace_dir."
        )
    reuse_corpus = bool(present_files)
    if reuse_corpus:
        manifest = json.loads((tokenized_path / "manifest.json").read_text(encoding="utf-8"))
        if any(manifest.get(key) != value for key, value in corpus_settings.items()):
            raise ValueError(
                f"Packed data in {tokenized_path} uses different settings. "
                "Restore the original settings to reuse it or choose a new workspace_dir."
            )
        print(f"Reusing packed data in {tokenized_path}.")

In [ ]:
def hugging_face_token():
    token = os.environ.get("HF_TOKEN", "").strip()
    if token:
        return token

    from google.colab import userdata

    try:
        return (userdata.get("HF_TOKEN") or "").strip() or None
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        return None


if train_model and not reuse_corpus:
    if min(story_limits.values()) < 1:
        raise ValueError("Story limits must be positive.")

    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")
    from datasets import load_dataset

    token = hugging_face_token()
    print("Downloading SimpleStories with Hugging Face authentication." if token else
          "Downloading public SimpleStories anonymously. Add HF_TOKEN in Colab Secrets if rate limited.")
    dataset = load_dataset(
        dataset_name,
        data_files={"train": SIMPLESTORIES_TRAIN_FILES},
        split="train",
        cache_dir=str(cache_dir),
        # The Hub metadata describes both splits, but only train shards were requested.
        verification_mode="no_checks",
        token=token,
    )

    del token
    rows = dataset.select_columns(["story"]).shuffle(seed=seed)
    cleaned = ((row["story"] or "").strip() for row in rows)
    usable = (story for story in cleaned if story and EOS not in story)

    validation = list(islice(usable, story_limits["val"]))
    validation_texts = set(validation)
    training = list(islice((story for story in usable if story not in validation_texts), story_limits["train"]))
    stories = {"train": training, "val": validation}
    for split, selected in stories.items():
        if not selected:
            raise ValueError(f"SimpleStories returned no usable stories for {split}.")
        print(f"Selected {len(selected):,} {split} stories.")

    for split, documents in stories.items():
        word_counts = [len(story.split()) for story in documents]
        p10, median, p90 = np.percentile(word_counts, [10, 50, 90])
        print(f"{split}: {len(documents):,} stories, {sum(word_counts):,} words")
        print(f"  Words per story: p10={p10:.0f}, median={median:.0f}, p90={p90:.0f}")

    print("\nExample training story\n")
    print(stories["train"][0])

## Train the tokenizer and save the corpus

Save the train and validation token files, tokenizer, and manifest under **data/tokenized**. Check EOS at every story boundary after packing. Existing artifacts are checked by hash before training.

In [ ]:
if train_model and not reuse_corpus:
    tokenizer = train_tokenizer(stories["train"], VOCAB_SIZE)
    preview = tokenizer.encode(stories["train"][0], add_special_tokens=False)

    print(f"Vocabulary size: {tokenizer.get_vocab_size():,}")
    print(f"First 25 tokens: {preview.tokens[:25]}")

In [ ]:
if train_model and not reuse_corpus:
    tokenized_path.mkdir(parents=True, exist_ok=True)
    for name in output_files:
        destination = tokenized_path / name
        if destination.exists():
            raise FileExistsError(f"Packed output already exists: {destination}")

    with TemporaryDirectory(prefix=".prepare-", dir=tokenized_path) as temporary:
        staging = Path(temporary)
        tokenizer_path = staging / "tokenizer.json"
        tokenizer_path.write_text(tokenizer.to_str(pretty=True), encoding="utf-8")

        manifest = {
            "version": 1,
            "dtype": TOKEN_DTYPE.str,
            **corpus_settings,
            "vocab_size": tokenizer.get_vocab_size(),
            "eos_id": tokenizer.token_to_id(EOS),
            "tokenizer_sha256": sha256(tokenizer_path),
        }
        for split, documents in stories.items():
            manifest[split] = pack_documents(staging / f"{split}.bin", tokenizer, iter(documents))

        (staging / "manifest.json").write_text(
            json.dumps(manifest, indent=2) + "\n", encoding="utf-8"
        )
        # Publish the manifest last so the trainer sees only a complete corpus.
        published = []
        try:
            for name in output_files:
                destination = tokenized_path / name
                (staging / name).rename(destination)
                published.append(destination)
        except BaseException:
            for destination in published:
                destination.unlink(missing_ok=True)
            raise

    for split in stories:
        print(f"{split}: {manifest[split]['tokens']:,} packed tokens")
    print(f"Saved to {tokenized_path}")

In [ ]:
if train_model and not reuse_corpus:
    saved_tokenizer = Tokenizer.from_file(str(tokenized_path / "tokenizer.json"))
    saved_manifest = json.loads((tokenized_path / "manifest.json").read_text(encoding="utf-8"))
    eos_id = saved_tokenizer.token_to_id(EOS)
    if eos_id is None or eos_id != saved_manifest["eos_id"]:
        raise ValueError("The saved tokenizer and manifest disagree on the EOS token.")


    for split in ("train", "val"):
        documents = stories[split]
        packed_path = tokenized_path / f"{split}.bin"
        split_info = saved_manifest[split]
        if not documents or len(documents) != split_info["documents"]:
            raise ValueError(f"{split}: selected stories do not match the saved document count.")
        if packed_path.stat().st_size != split_info["tokens"] * TOKEN_DTYPE.itemsize:
            raise ValueError(f"{split}: packed file size does not match the saved token count.")

        tokens = np.memmap(packed_path, dtype=TOKEN_DTYPE, mode="r")
        offset = 0
        for start in range(0, len(documents), 2048):
            batch = documents[start : start + 2048]
            encoded = saved_tokenizer.encode_batch(batch, add_special_tokens=False)
            lengths = [len(story.ids) + 1 for story in encoded]
            eos_positions = offset + np.cumsum(lengths) - 1
            if eos_positions[-1] >= len(tokens) or np.any(tokens[eos_positions] != eos_id):
                raise ValueError(f"{split}: missing EOS in the batch starting at story {start + 1:,}.")
            offset = int(eos_positions[-1]) + 1

        if offset != len(tokens) or np.count_nonzero(tokens == eos_id) != len(documents):
            raise ValueError(f"{split}: expected exactly one EOS per story and EOS at the end of the file.")
        print(f"{split}: verified EOS after all {len(documents):,} stories, including the final token.")
        del tokens

## Define the Transformer

The base preset uses a 256-token context, 512-dimensional embeddings, 10 blocks, and 8 causal attention heads. The output projection shares its weights with the token embedding.

In [ ]:
from dataclasses import dataclass
import math

from torch import nn
from torch.nn import functional as F


@dataclass(frozen=True)
class GPTConfig:
    vocab_size: int
    context_length: int = 256
    n_layer: int = 10
    n_head: int = 8
    n_embd: int = 512

    def __post_init__(self):
        if min(self.vocab_size, self.context_length, self.n_layer, self.n_head, self.n_embd) < 1:
            raise ValueError("Model dimensions must be positive")
        if self.n_embd % self.n_head:
            raise ValueError("n_embd must be divisible by n_head")


class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.n_head = config.n_head
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, length, n_embd = x.shape
        qkv = self.c_attn(x).reshape(batch, length, 3, self.n_head, n_embd // self.n_head)
        query, key, value = qkv.permute(2, 0, 3, 1, 4).unbind()
        attended = F.scaled_dot_product_attention(query, key, value, is_causal=True)
        return self.c_proj(attended.transpose(1, 2).reshape(batch, length, n_embd))


class MLP(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.c_proj(self.gelu(self.c_fc(x)))


class Block(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln_1(x))
        return x + self.mlp(self.ln_2(x))


class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict({
            "wte": nn.Embedding(config.vocab_size, config.n_embd),
            "wpe": nn.Embedding(config.context_length, config.n_embd),
            "h": nn.ModuleList(Block(config) for _ in range(config.n_layer)),
            "ln_f": nn.LayerNorm(config.n_embd),
        })
        self.apply(self._init_weights)
        residual_std = 0.02 / math.sqrt(2 * config.n_layer)
        for block in self.transformer.h:
            nn.init.normal_(block.attn.c_proj.weight, std=residual_std)
            nn.init.normal_(block.mlp.c_proj.weight, std=residual_std)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.transformer.wte.weight

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        length = input_ids.size(1)
        if length > self.config.context_length:
            raise ValueError("Input exceeds the configured context length")
        positions = torch.arange(length, device=input_ids.device)
        token_embeddings = self.transformer.wte(input_ids)
        position_embeddings = self.transformer.wpe(positions)
        hidden = token_embeddings + position_embeddings
        for block in self.transformer.h:
            hidden = block(hidden)
        return self.lm_head(self.transformer.ln_f(hidden))

## Define the training helpers

Read packed next-token windows, schedule the learning rate, evaluate on held-out windows, and save or load checkpoints.

In [ ]:
@dataclass(frozen=True)
class TrainConfig:
    batch_size: int
    steps: int
    warmup_steps: int
    learning_rate: float
    min_learning_rate: float
    weight_decay: float
    eval_every: int
    eval_windows: int
    seed: int

    def __post_init__(self):
        if min(self.batch_size, self.steps, self.eval_every, self.eval_windows) < 1:
            raise ValueError(
                "Batch, step, and evaluation counts must be positive")
        if not 0 <= self.warmup_steps < self.steps:
            raise ValueError("Warmup must be shorter than the run")
        if not 0 <= self.min_learning_rate <= self.learning_rate or self.learning_rate == 0:
            raise ValueError(
                "Require 0 <= min_learning_rate <= learning_rate and a positive peak")
        if self.weight_decay < 0:
            raise ValueError("Weight decay cannot be negative")

    def lr_at(self, step: int) -> float:
        if step <= self.warmup_steps:
            return self.learning_rate * step / self.warmup_steps
        decay_intervals = max(1, self.steps - self.warmup_steps - 1)
        progress = (step - self.warmup_steps - 1) / decay_intervals
        cosine = 0.5 * (1 + math.cos(math.pi * progress))
        return self.min_learning_rate + cosine * (self.learning_rate - self.min_learning_rate)


class TokenStream:
    def __init__(self, path: Path, size: int, context: int):
        with path.open("rb") as stream:
            self.tokens = np.memmap(
                stream, dtype=TOKEN_DTYPE, mode="r", shape=(size,))
        self.offsets = np.arange(context + 1)
        self.windows = (size - 1) // context
        if self.windows < 1:
            raise ValueError(f"{path} needs at least {context + 1} tokens")

    def batch(self, starts: np.ndarray, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
        batch = self.tokens[starts[:, None] + self.offsets]
        ids = torch.from_numpy(batch.astype(np.int64)).to(device)
        return ids[:, :-1].contiguous(), ids[:, 1:].contiguous()


def loss_for(model, inputs, targets, precision):
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=precision == "bf16"):
        logits = model(inputs)
        return F.cross_entropy(logits.flatten(0, 1), targets.flatten())


@torch.inference_mode()
def evaluate(model, data, config, device, precision) -> float:
    model.eval()
    count = min(data.windows, config.eval_windows)
    windows = np.linspace(0, data.windows - 1, num=count, dtype=np.int64)
    starts = windows * model.config.context_length
    total = torch.zeros((), device=device)
    for chunk in range(0, count, config.batch_size):
        offsets = starts[chunk: chunk + config.batch_size]
        x, y = data.batch(offsets, device)
        total += loss_for(model, x, y, precision) * len(offsets)
    model.train()
    return total.item() / count


def save_checkpoint(path: Path, state: dict) -> None:
    with TemporaryDirectory(prefix=".checkpoint-", dir=path.parent) as temporary:
        staging = Path(temporary) / path.name
        with staging.open("wb") as stream:
            torch.save(state, stream)
        staging.replace(path)


def load_checkpoint(path: Path) -> dict:
    with path.open("rb") as stream:
        checkpoint = torch.load(stream, map_location="cpu", weights_only=True)
    config = checkpoint.get("model_config", {})
    for old_name, new_name in (
        ("context", "context_length"),
        ("block_size", "context_length"),
        ("layers", "n_layer"),
        ("heads", "n_head"),
        ("width", "n_embd"),
        ("embedding_dim", "n_embd"),
    ):
        if old_name in config:
            config.setdefault(new_name, config.pop(old_name))

    # Keep old model weights and optimizer parameter order compatible after renaming.
    aliases = {
        "tokens.weight": "transformer.wte.weight",
        "embedding_table.weight": "transformer.wte.weight",
        "positions.weight": "transformer.wpe.weight",
        "position_embedding_table.weight": "transformer.wpe.weight",
        "norm.weight": "transformer.ln_f.weight",
        "norm.bias": "transformer.ln_f.bias",
    }
    state = checkpoint["model"]
    for name in list(state):
        renamed = aliases.get(name, name)
        if name.startswith("blocks."):
            renamed = "transformer.h." + name.removeprefix("blocks.")
            for old_name, new_name in (
                (".attention_norm.", ".ln_1."),
                (".attention.qkv.", ".attn.c_attn."),
                (".attention.out.", ".attn.c_proj."),
                (".mlp_norm.", ".ln_2."),
                (".mlp.0.", ".mlp.c_fc."),
                (".mlp.2.", ".mlp.c_proj."),
            ):
                renamed = renamed.replace(old_name, new_name)
        if renamed != name:
            state[renamed] = state.pop(name)
    state.setdefault("lm_head.weight", state["transformer.wte.weight"])
    return checkpoint

## Load and check the packed corpus

Check the file hashes and split provenance before opening the training and validation streams. The model uses the vocabulary size recorded in the manifest.

In [ ]:
if train_model:
    settings = PRESETS[preset]
    manifest = json.loads((tokenized_path / "manifest.json").read_text(encoding="utf-8"))
    if manifest["version"] != 1 or manifest["dtype"] != TOKEN_DTYPE.str:
        raise ValueError("Unsupported dataset format")
    if manifest.get("dataset") == "SimpleStories/SimpleStories" and manifest.get("source_splits") != {
        "train": "train",
        "val": "train",
    }:
        raise ValueError("Prepare SimpleStories with the current notebook so both splits come from the published training data")

    tokenizer_path = tokenized_path / "tokenizer.json"
    if sha256(tokenizer_path) != manifest["tokenizer_sha256"]:
        raise ValueError("Tokenizer does not match the dataset manifest")

    tokenizer_json = tokenizer_path.read_text(encoding="utf-8")
    vocab_size = manifest["vocab_size"]
    model_config = GPTConfig(vocab_size=vocab_size, **settings["model"])
    print(f"Model: n_embd={model_config.n_embd:,}, n_head={model_config.n_head}, "
          f"head_dim={model_config.n_embd // model_config.n_head}")
    config = TrainConfig(**settings["training"])
    stop = config.steps if stop_after is None else stop_after
    if not 1 <= stop <= config.steps:
        raise ValueError("stop_after must be between 1 and the configured total steps")

    streams = {}
    for split in ("train", "val"):
        path = tokenized_path / f"{split}.bin"
        size = manifest[split]["tokens"]
        if path.stat().st_size != size * TOKEN_DTYPE.itemsize or sha256(path) != manifest[split]["sha256"]:
            raise ValueError(f"{path} does not match the dataset manifest")
        streams[split] = TokenStream(path, size, model_config.context_length)

    for split in ("train", "val"):
        print(f"{split}: {manifest[split]['documents']:,} stories, {manifest[split]['tokens']:,} tokens")

## Batch and epoch analysis

A full pass below means taking non-overlapping windows of the configured context length. The final batch of that hypothetical pass may be smaller. Training samples random windows with replacement, so its epoch count measures how many token targets it processes relative to the size of the corpus. Validation checks the same evenly spaced windows each time.

In [ ]:
if train_model:
    context_length = model_config.context_length
    batch_size = config.batch_size
    targets_per_train_step = batch_size * context_length

    print(f"{preset.title()} preset: {batch_size:,} windows per batch, {context_length:,} targets per window")
    for split, label in (("train", "Training"), ("val", "Validation")):
        token_count = manifest[split]["tokens"]
        windows = streams[split].windows
        full_batches, last_batch_size = divmod(windows, batch_size)
        pass_batches = full_batches + int(last_batch_size > 0)
        tail_targets = (token_count - 1) % context_length
        print(f"{label}: {token_count:,} packed tokens and {windows:,} non-overlapping windows")
        batch_breakdown = f"{full_batches:,} full batches"
        if last_batch_size:
            batch_breakdown += f" + {last_batch_size:,} windows in the final batch"
        print(f"  One pass: {pass_batches:,} batches ({batch_breakdown})")
        print(f"  Tail outside the full windows: {tail_targets:,} targets")

    train_targets = manifest["train"]["tokens"] - 1
    scheduled_targets = stop * targets_per_train_step
    print(f"\nTraining from scratch through step {stop:,}: "
          f"{scheduled_targets:,} targets = {scheduled_targets / train_targets:.4f} epoch equivalents")

    val_windows = min(config.eval_windows, streams["val"].windows)
    val_batches = (val_windows + batch_size - 1) // batch_size
    val_fraction = val_windows * context_length / (manifest["val"]["tokens"] - 1)
    fresh_run_checks = 1 + (stop + config.eval_every - 1) // config.eval_every
    batch_label = "batch" if val_batches == 1 else "batches"
    print(f"Validation per check: {val_windows:,} windows in {val_batches:,} {batch_label}, "
          f"covering {val_fraction:.2%} of validation targets ({val_fraction:.4f} of one pass)")
    print(f"Fresh-run schedule through step {stop:,}: {fresh_run_checks:,} checks, including step 0")

## Initialize or resume training

To pause at an absolute optimizer step, set **stop_after** in the settings cell before starting. To resume, set **resume = True**, reset **stop_after = None** to finish the scheduled run, and rerun from the settings cell. Use the same preset, workspace, and packed data. With Drive enabled, this also works in a fresh runtime after installing the dependencies.

In [ ]:
if train_model:
    if resume and not checkpoint_path.is_file():
        raise FileNotFoundError(f"No checkpoint to resume at {checkpoint_path}.")

    if not resume and checkpoint_path.exists():
        raise FileExistsError(
            f"Checkpoint already exists: {checkpoint_path}. Set resume = True to continue.")

    torch.manual_seed(config.seed)
    batch_rng = torch.Generator(device="cpu").manual_seed(config.seed)
    model = GPT(model_config).to(device)
    decay, no_decay = [], []

    for parameter in model.parameters():
        (decay if parameter.ndim >= 2 else no_decay).append(parameter)

    optimizer = torch.optim.AdamW(
        [{"params": decay}, {"params": no_decay, "weight_decay": 0.0}],
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
        betas=(0.9, 0.95),
        fused=True,
    )
    tokens_per_step = config.batch_size * model_config.context_length
    run_identity = {
        "model_config": asdict(model_config),
        "train_config": asdict(config),
        "data_manifest": manifest,
    }
    start = 0
    if resume:
        checkpoint = load_checkpoint(checkpoint_path)
        if checkpoint["version"] != 1 or any(checkpoint.get(k) != v for k, v in run_identity.items()):
            raise ValueError(
                "Resume requires matching training settings and data. "
                "For a fresh run, select a new checkpoint path and set resume=False."
            )
        start = checkpoint["step"]
        if stop <= start:
            raise ValueError(
                f"stop_after must exceed the completed step ({start})")
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        batch_rng.set_state(checkpoint["batch_rng"])
        history = checkpoint["metrics"]
        del checkpoint
    else:
        history = []

    training_started = False
    num_parameters = sum(parameter.numel() for parameter in model.parameters())
    print(
        f"{num_parameters:,} parameters | {device} {precision} | "
        f"{tokens_per_step:,} tokens/step | steps {start + 1} to {stop}",
        flush=True,
    )

## Train and validate

Each optimizer step runs one batch of next-token predictions. The loop logs training and validation loss and saves a resumable checkpoint at each evaluation interval and at the requested stop step. After an interruption, use the resume settings to continue from the last saved checkpoint.

In [ ]:
if train_model:
    if training_started:
        raise RuntimeError("Rerun the settings and initialization cells before starting training again.")

    if not resume:
        run_dir.mkdir(parents=True, exist_ok=True)
        if checkpoint_path.exists():
            raise FileExistsError(f"Checkpoint already exists: {checkpoint_path}")

    training_started = True
    if not resume:
        initial_loss = evaluate(model, streams["val"], config, device, precision)
        history = [{"step": 0, "tokens": 0, "val_loss": initial_loss}]
        print(f"step 0 | val loss {initial_loss:.4f}", flush=True)

    interval_start = perf_counter()
    interval_steps = 0
    running_loss = torch.zeros((), device=device)
    for step in range(start + 1, stop + 1):
        lr = config.lr_at(step)
        for group in optimizer.param_groups:
            group["lr"] = lr
        optimizer.zero_grad(set_to_none=True)
        starts = torch.randint(
            len(streams["train"].tokens) - model_config.context_length,
            (config.batch_size,),
            generator=batch_rng,
            device="cpu",
        ).numpy()
        x, y = streams["train"].batch(starts, device)
        loss = loss_for(model, x, y, precision)
        running_loss += loss.detach()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        interval_steps += 1

        if step % config.eval_every != 0 and step != stop:
            continue
        train_loss = running_loss.item() / interval_steps
        throughput = interval_steps * tokens_per_step / (perf_counter() - interval_start)
        val_loss = evaluate(model, streams["val"], config, device, precision)
        if not math.isfinite(train_loss) or not math.isfinite(val_loss):
            raise FloatingPointError("Non-finite loss. The last valid checkpoint is preserved.")
        metrics = {
            "step": step,
            "tokens": step * tokens_per_step,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "lr": lr,
            "train_tokens_per_second": round(throughput),
        }
        history.append(metrics)
        save_checkpoint(
            checkpoint_path,
            {
                "version": 1,
                **run_identity,
                "step": step,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "batch_rng": batch_rng.get_state(),
                "tokenizer_json": tokenizer_json,
                "metrics": history,
            },
        )
        print(json.dumps(metrics), flush=True)
        running_loss.zero_()
        interval_steps = 0
        interval_start = perf_counter()
    print(f"Checkpoint saved to {checkpoint_path}", flush=True)

## Generate a story continuation

For inference only, set **train_model = False** and run the notebook with a saved checkpoint. Enable Drive to use a saved checkpoint there, or set the checkpoint path in the settings cell to the path of the file you uploaded to Colab. The default **inference_device = "cpu"** lets you generate text on CPU. Set it to **"cuda"** for GPU inference. Edit the prompt and token limit in the cell below.

In [ ]:
@torch.inference_mode()
def complete(checkpoint_path: Path, prompt: str, max_new_tokens: int, device: torch.device) -> str:
    checkpoint = load_checkpoint(checkpoint_path)
    tokenizer = Tokenizer.from_str(checkpoint["tokenizer_json"])
    config = GPTConfig(**checkpoint["model_config"])
    model = GPT(config).to(device).eval()
    model.load_state_dict(checkpoint["model"])
    del checkpoint
    ids = tokenizer.encode(prompt, add_special_tokens=False).ids
    if not ids or len(ids) > config.context_length:
        raise ValueError(f"Prompt must encode to between 1 and {config.context_length} tokens")
    generated = []
    eos_id = tokenizer.token_to_id(EOS)
    for _ in range(max_new_tokens):
        x = torch.tensor([ids[-config.context_length :]], device=device)
        next_id = model(x)[0, -1].argmax().item()
        if next_id == eos_id:
            break
        ids.append(next_id)
        generated.append(next_id)
    return tokenizer.decode(generated)

In [ ]:
prompt = "Once upon a time, there was a little"
max_new_tokens = 80

completion = complete(checkpoint_path, prompt, max_new_tokens, torch.device(inference_device))
print(prompt + completion)

With Drive enabled, find the packed data and checkpoint in **MyDrive/MiniReasoner**. If you used runtime storage, download the checkpoint and the four files under **data/tokenized** from Colab's Files pane before ending the session if you want to resume elsewhere.